# Smoke Test — Issue 9: Verdict Generation & Recommendation Layer

End-to-end test of the two-stage verdict system:
- **Stage 1** — `match_rule()` deterministic decision tree maps forensics signals → `RecommendationRule`
- **Stage 2** — `render_recommendation()` calls Claude to render the matched rule as one concrete sentence

**Prerequisites:**
- `backend/.env` contains `ANTHROPIC_API_KEY`
- ChromaDB index exists at `backend/data/` (run `bootstrap_data.py` if not)
- Run from repo root: `cd backend && poetry run uvicorn main:app --port 8001`

**Acceptance criteria being verified:** all 13 listed at the bottom of this notebook.

In [ ]:
import requests
import json
import subprocess
import time

BASE = "http://localhost:8001"

# Quick liveness check — if this fails, start the server first:
# cd backend && poetry run uvicorn main:app --port 8001
resp = requests.get(f"{BASE}/docs")
print("Server status:", resp.status_code, "(200 = up)")

## 1. Response shape — `recommendation` and `rule_id` present

In [ ]:
# Load an example then run the full analyze pipeline
example = requests.post(f"{BASE}/example", json={"domain": "techqa"}).json()
print("Example ID :", example["example_id"])
print("Question   :", example["question"][:100])
print("Context    :", example["context_preview"][:120], "...")

Expected output (2026-07-08 run):
```
Example ID : techqa_TRAIN_Q414
Question   : Is MQ V7.0 is going out of support? Is Websphere MQ V7.0 is going out of support?
Context    :  NEWS

ABSTRACT
 IBM has withdrawn support for WebSphere MQ V7.0.1 ...
```

In [ ]:
result = requests.post(f"{BASE}/analyze", json={"example_id": example["example_id"]}).json()

# Verify all 13 expected keys are present
EXPECTED_KEYS = [
    "question", "generated_answer", "retrieved_chunks",
    "ragas", "retrieval_score_distribution", "hedging_mismatch",
    "chunk_attribution", "confidence_calibration", "retrieval_distribution",
    "embedding_space", "query_corpus_fit",
    "recommendation", "rule_id",   # <-- issue 9 adds these two
]
actual_keys = list(result.keys())
missing = [k for k in EXPECTED_KEYS if k not in actual_keys]

print("Keys present :", actual_keys)
print("Missing keys :", missing or "none — all present ✓")

Expected output:
```
Keys present : ['question', 'generated_answer', 'retrieved_chunks', 'ragas',
                'retrieval_score_distribution', 'hedging_mismatch', 'chunk_attribution',
                'confidence_calibration', 'retrieval_distribution', 'embedding_space',
                'query_corpus_fit', 'recommendation', 'rule_id']
Missing keys : none — all present ✓
```

## 2. Verdict fields — rule fired and recommendation rendered

In [ ]:
print("rule_id        :", result["rule_id"])
print("recommendation :", result["recommendation"])
print()
print("Word count     :", len(result["recommendation"].split()), "words (must be ≤ 50)")
assert len(result["recommendation"].split()) <= 50, "recommendation exceeds 50 words!"
print("Word count     : ✓")

Expected output:
```
rule_id        : R08
recommendation : Your query phrasing mismatches the embedding space, causing ambiguous retrieval
                 despite relevant content existing in the corpus — rephrase the user query using
                 terminology closer to how the source material is written to unlock accurate retrieval.

Word count     : 43 words (must be ≤ 50)
Word count     : ✓
```

## 3. Signals that triggered R08 — verify the decision tree path

In [ ]:
# R08 fires when query_corpus_fit.triggered=True and mismatch_type="query_mismatch".
# query_corpus_fit is triggered when retrieval_relevance_score < 0.5 OR
# query_isolation > 1.2 OR (score_entropy > 1.5 AND faithfulness < 0.5).

qf = result["query_corpus_fit"]
rd = result["retrieval_distribution"]
ragas = result["ragas"]

print("query_corpus_fit.triggered    :", qf["triggered"])
print("query_corpus_fit.mismatch_type:", qf["mismatch_type"])
print()
print("--- Trigger signals ---")
print(f"retrieval_relevance_score  : {ragas['retrieval_relevance_score']:.3f}  (< 0.5 triggers fit)")
print(f"score_entropy              : {rd['score_entropy']:.3f}  (> 1.5 with low faithfulness triggers fit)")
print(f"faithfulness_score         : {ragas['faithfulness_score']:.3f}")

assert qf["triggered"] is True
assert qf["mismatch_type"] == "query_mismatch"
print()
print("R08 trigger path verified ✓")

Expected output:
```
query_corpus_fit.triggered    : True
query_corpus_fit.mismatch_type: query_mismatch

--- Trigger signals ---
retrieval_relevance_score  : 0.000  (< 0.5 triggers fit)
score_entropy              : 1.608  (> 1.5 with low faithfulness triggers fit)
faithfulness_score         : 0.917

R08 trigger path verified ✓
```

## 4. Three domains — rule fires consistently, recommendation varies per run

In [ ]:
# Stage 2 uses Claude, so the exact wording varies per call.
# We assert structure (rule_id present, word count, non-empty) not exact text.

for domain in ["techqa", "finqa", "covidqa"]:
    ex = requests.post(f"{BASE}/example", json={"domain": domain}).json()
    r  = requests.post(f"{BASE}/analyze", json={"example_id": ex["example_id"]}).json()

    rule_id = r["rule_id"]
    rec     = r["recommendation"]
    words   = len(rec.split())
    faith   = r["ragas"]["faithfulness_score"]
    relev   = r["ragas"]["retrieval_relevance_score"]

    assert rule_id, "rule_id is empty!"
    assert words <= 50, f"recommendation too long: {words} words"
    assert len(rec) > 10, "recommendation suspiciously short"

    print(f"[{domain:8s}] {ex['example_id']:30s}  rule={rule_id}  faith={faith:.2f}  relev={relev:.2f}  ({words}w)")
    print(f"           {rec[:120]}")
    print()

Expected output (2026-07-08 run — exact wording will differ):
```
[techqa  ] techqa_TRAIN_Q414                rule=R08  faith=0.92  relev=0.00  (43w)
           Your query phrasing mismatches the embedding space, causing ambiguous retrieval despite
           relevant content existing in the corpus — rephrase the user query using terminology...

[finqa   ] finqa_4438                       rule=R08  faith=0.89  relev=0.00  (45w)
           The query phrasing mismatches the embedding space despite relevant corpus content being
           present — rephrase the user query using terminology closer to how the source material...

[covidqa ] 1341                             rule=R08  faith=1.00  relev=0.00  (47w)
           Your query isn't matching the embedding space despite relevant content existing in the
           corpus — rephrase the user query using terminology closer to the corpus language...
```

## 5. Unit test suite — all 15 pass

In [ ]:
import subprocess

result_proc = subprocess.run(
    ["poetry", "run", "pytest", "tests/test_verdict_generator.py", "-v", "--tb=short", "--no-header"],
    capture_output=True, text=True,
    cwd="../backend"
)
print(result_proc.stdout[-3000:])  # last 3000 chars — all results
assert result_proc.returncode == 0, "Tests failed!"

Expected output:
```
tests/test_verdict_generator.py::test_r01_high_entropy_low_faithfulness PASSED
tests/test_verdict_generator.py::test_r01_high_entropy_high_chunk_spread PASSED
tests/test_verdict_generator.py::test_r02_high_gap_high_unattributed PASSED
tests/test_verdict_generator.py::test_r03a_decisive_retrieval_wrong_content PASSED
tests/test_verdict_generator.py::test_r03b_high_gap_high_query_isolation PASSED
tests/test_verdict_generator.py::test_r04_low_decay_overconfident PASSED
tests/test_verdict_generator.py::test_r05_high_tail_mass_high_weak_match PASSED
tests/test_verdict_generator.py::test_r06_strong_retrieval_underconfident PASSED
tests/test_verdict_generator.py::test_r07_healthy_pipeline PASSED
tests/test_verdict_generator.py::test_r08_query_mismatch PASSED
tests/test_verdict_generator.py::test_r09_coverage_gap PASSED
tests/test_verdict_generator.py::test_r08_takes_priority_over_r01 PASSED
tests/test_verdict_generator.py::test_render_recommendation_under_50_words PASSED
tests/test_verdict_generator.py::test_render_recommendation_claude_failure_falls_back_to_action PASSED
tests/test_verdict_generator.py::test_match_rule_never_returns_none PASSED

============================== 15 passed in 0.13s ==============================
```

## Acceptance criteria

| # | Criterion | Verified by |
|---|-----------|-------------|
| 1 | `match_rule()` accepts all forensics metric objects incl. `QueryCorpusFitMetrics` | Unit test 1–12 |
| 2 | `match_rule()` always returns R07 when all signals healthy — never crashes/None | Unit test 15 |
| 3 | Every rule R01–R09 reachable by some input combination | Unit tests 1–11 |
| 4 | R08 returned when `triggered=True` and `mismatch_type="query_mismatch"` | Unit test 10 + live §3 |
| 5 | R09 returned when `triggered=True` and `mismatch_type="coverage_gap"` | Unit test 11 |
| 6 | R08/R09 checked before R01–R06 | Unit test 12 + code inspection |
| 7 | `render_recommendation()` returns string under 50 words | Unit test 13 + live §2, §4 |
| 8 | Rendered recommendation names specific component and action | Live §2, §4 |
| 9 | All prompts in `prompts/verdict_prompts.py` — no inline strings | Code inspection |
| 10 | All rules in `prompts/recommendation_rules.py` — no inline strings | Code inspection |
| 11 | `recommendation` and `rule_id` on `AnalyzeResponse` | Live §1 |
| 12 | Claude failure falls back to `rule.action` — never crashes | Unit test 14 |
| 13 | All thresholds in `match_rule()` have empirical-basis comments | Code inspection |